Specify the model path and load the tokenizer.

## NOTE: 

Reference to read on quantizing giant LLMs. --> [Multiplication for transformers at scale using Hugging Face Transformers, Accelerate and bitsandbytes](https://huggingface.co/blog/hf-bitsandbytes-integration)

In [1]:

MODEL_PATH = "/teamspace/studios/this_studio/hf-models/mistral/pytorch/7b-instruct-v0.1-hf/1"

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # For debugging CUDA errors
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Specify which GPU to use
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disable parallelism in tokenizers to avoid warnings

In [3]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

/teamspace/studios/this_studio/virtual_envs/hf_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Clear torch cache and check for available devices.

In [4]:

with torch.no_grad():
    torch.cuda.empty_cache()
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}, {torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'}")
print(f"Total memory: {device}, {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB" if device.type == "cuda" else "N/A ")

Using device: cuda, Tesla T4
Total memory: cuda, 15.64 GB



Load the Mistral 7B Instruct model and tokenizer.

In [5]:
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH).to(device) 
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 11224.41it/s]



Define a conversation with the model using the chat template and encode the messages for input to the model.

In [6]:

def print_messages(messages):
    """Helper function to print messages in a readable format."""
    for msg in messages:
        print(f"{msg['role'].upper()}: {msg['content']}\n")
        
def model_chat(messages):
    """Function to generate a response from the model given a list of messages in the chat format."""
    print_messages(messages)
    encodeds = tokenizer.apply_chat_template(messages, return_tensors="pt")
    
    model_inputs = encodeds.to(device)
    
    generated_ids = model.generate( #ignore: pylint: disable=E1120
        input_ids=model_inputs.input_ids, 
        attention_mask=model_inputs.attention_mask,
        max_new_tokens=10000, 
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_p=0.9, 
        temperature=0.7)
    decoded = tokenizer.batch_decode(generated_ids,
                                     skip_special_tokens=True)
    return f"MODEL RESPONSE:\n{decoded[0]}\n"    


Example conversation with the model.

In [7]:
messages = [
    {"role": "user", "content": "What is your favorite condiment?"},
    {"role": "assistant", "content": "Well, I'm quite partial to a good squeeze of fresh lemon juice. It adds just the right amount of zesty flavor to whatever I'm cooking up in the kitchen"},
    {"role": "user", "content": "Do you have a mayonnaise receipe?"}
]


Generate responses from model.

In [8]:
response = model_chat(messages)
print(response)

USER: What is your favorite condiment?

ASSISTANT: Well, I'm quite partial to a good squeeze of fresh lemon juice. It adds just the right amount of zesty flavor to whatever I'm cooking up in the kitchen

USER: Do you have a mayonnaise receipe?

MODEL RESPONSE:
[INST] What is your favorite condiment? [/INST]Well, I'm quite partial to a good squeeze of fresh lemon juice. It adds just the right amount of zesty flavor to whatever I'm cooking up in the kitchen  [INST] Do you have a mayonnaise receipe? [/INST] Certainly! Here's a simple recipe for homemade mayonnaise that you can whip up in no time:

Ingredients:

* 1 egg yolk
* 1 tablespoon Dijon mustard
* 1 tablespoon apple cider vinegar
* 1/2 teaspoon salt
* 1/4 cup olive oil
* 1/4 cup vegetable oil

Instructions:

1. In a small bowl, whisk together the egg yolk, mustard, apple cider vinegar, and salt until well combined.
2. Slowly drizzle in the olive oil and vegetable oil while whisking constantly until the mixture thickens and emulsifi


Generate a receipe for a vegetarian dish.

In [9]:

messages = [
    {"role": "user", 
     "content": "Act as a gourmet chef. \
        I have a friend coming over who is a vegetarian. \
        Can you give me a recipe for a vegetarian dish that is easy to make and delicious?"
    }
]

response = model_chat(messages)
print(response)

USER: Act as a gourmet chef.         I have a friend coming over who is a vegetarian.         Can you give me a recipe for a vegetarian dish that is easy to make and delicious?

MODEL RESPONSE:
[INST] Act as a gourmet chef.         I have a friend coming over who is a vegetarian.         Can you give me a recipe for a vegetarian dish that is easy to make and delicious? [/INST] Of course! Here's a recipe for a delicious and easy-to-make vegetable stir-fry that your friend will love.

Ingredients:

* 2 tablespoons of vegetable oil
* 1 tablespoon of minced garlic
* 1 tablespoon of grated ginger
* 1 sliced onion
* 1 sliced bell pepper
* 1 sliced carrot
* 1 sliced zucchini
* 1 cup of sliced mushrooms
* 1 cup of broccoli florets
* 2 tablespoons of soy sauce
* 1 tablespoon of rice vinegar
* 1 tablespoon of honey
* 1 tablespoon of cornstarch
* Salt and pepper to taste
* Cooked rice or noodles for serving

Instructions:

1. Heat the vegetable oil in a wok or large frying pan over medium-high he